# Qwen3 1.7B Local Test Baseline

Closed-book baseline evaluation for `Qwen/Qwen3-1.7B` on all 60 test problems.

## Setup

Run this locally from VS Code. The model downloads into the Hugging Face cache, not this repository.

In [ ]:
!pip install -q -U transformers accelerate pandas tqdm python-dotenv

In [ ]:
from pathlib import Path
import os
import sys

import torch
from dotenv import load_dotenv
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
PROJECT_ROOT

In [ ]:
from training_eval.eval_utils import (
    default_test_dir,
    extract_json_object,
    is_correct,
    load_jsonl_records,
    make_closed_book_prompt,
    rows_to_frame,
    save_results,
    summarize_accuracy,
)

## Load Dev Records

In [ ]:
DATA_DIR = default_test_dir(PROJECT_ROOT)
records = load_jsonl_records(DATA_DIR, pattern="*_preview.jsonl")
len(records), DATA_DIR

## Load Model

Qwen3 defaults to a thinking-style chat mode. The generation helper below sets `enable_thinking=False` so the model has a much better chance of returning the requested final JSON inside the token budget.


In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B"
HF_TOKEN = os.environ.get("HF_TOKEN")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float16 if device.type == "mps" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=HF_TOKEN, torch_dtype=dtype)
model.to(device)
model.eval()

device

In [ ]:
MAX_NEW_TOKENS = 256


def generate_answer(problem):
    messages = [{"role": "user", "content": make_closed_book_prompt(problem)}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


## One-Example Smoke Test

In [ ]:
raw_output = generate_answer(records[0]["problem"])
predicted = extract_json_object(raw_output)
raw_output, predicted, records[0]["canonical_answer"], is_correct(predicted, records[0]["canonical_answer"])

## Run Full Test Evaluation

In [ ]:
rows = []

for record in tqdm(records):
    raw_output = generate_answer(record["problem"])
    predicted = extract_json_object(raw_output)
    metadata = record.get("metadata", {})

    rows.append({
        "id": record["id"],
        "family": record["family"],
        "problem_type": record["problem_type"],
        "difficulty": record["difficulty"],
        "manual_variation": metadata.get("manual_variation", False),
        "manual_problem_variation": metadata.get("manual_problem_variation", False),
        "manual_reasoning_variation": metadata.get("manual_reasoning_variation", False),
        "problem": record["problem"],
        "canonical_answer": record["canonical_answer"],
        "raw_output": raw_output,
        "predicted_answer": predicted,
        "correct": is_correct(predicted, record["canonical_answer"]),
    })

df = rows_to_frame(rows)
df.head()

## Metrics

In [ ]:
print(f"Overall accuracy: {df['correct'].mean():.3f} ({df['correct'].sum()}/{len(df)})")
display(df.groupby("family")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby("difficulty")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby("manual_variation")["correct"].agg(["mean", "sum", "count"]).sort_index())

## Save Results

In [ ]:
result_dir = PROJECT_ROOT / "results" / "baselines" / "qwen3_1_7b_dev_closed_book"
metrics = summarize_accuracy(df)
metrics.update({
    "model": MODEL_NAME,
    "provider": "local_transformers",
    "dataset": "benchmark/data/test/*.jsonl",
})

outputs_path, metrics_path, csv_path = save_results(rows, result_dir, metrics)
outputs_path, metrics_path, csv_path

## Inspect Mistakes

In [ ]:
df.loc[~df["correct"], ["id", "family", "problem_type", "difficulty", "canonical_answer", "predicted_answer", "raw_output"]].head(20)